# Tema de Programare: Construirea unui Clasificator Naive Bayes de la Zero

Bun venit la tema despre clasificarea Naive Bayes.

In exemplele anterioare ai lucrat cu modele de machine learning de tip black-box din scikit-learn. Desi acestea sunt puternice si convenabile, intelegerea modului in care functioneaza "sub capota" este esentiala pentru a deveni un practician priceput in machine learning. Naive Bayes este unul dintre cei mai eleganti si mai interpretabili algoritmi din machine learning, bazat pe teoria probabilitatilor si pe teorema lui Bayes.

In aceasta tema, vei implementa clasificatori Naive Bayes de la zero folosind doar NumPy. Vei lucra atat cu trasaturi continue (Gaussian Naive Bayes), cat si cu trasaturi discrete (Multinomial Naive Bayes), aplicandu-le la probleme reale de clasificare, inclusiv detectarea spam-ului.

Naive Bayes face presupunerea "naive" ca trasaturile sunt independente conditionat de eticheta clasei. In ciuda acestei simplificari, algoritmul are performante remarcabil de bune in multe aplicatii practice, mai ales in clasificarea textelor, diagnostic medical si filtrarea spam-ului.

**Ce vei face in aceasta tema**

* Vei intelege fundamentul matematic al lui Naive Bayes folosind teorema lui Bayes.
* Vei implementa Gaussian Naive Bayes pentru trasaturi continue de la zero.
* Vei calcula probabilitati de verosimilitate folosind distributia Gaussiana.
* Vei aplica teorema lui Bayes pentru a face predictii cu distributii de probabilitate.
* Vei implementa calibrarea probabilitatilor pentru a imbunatati increderea predictiilor.
* Vei construi Multinomial Naive Bayes pentru clasificarea textelor si trasaturi discrete.
* Vei compara Naive Bayes cu Regresia Logistica pentru a intelege cand exceleaza fiecare.
* Vei crea un clasificator complet de spam cu scoruri de probabilitate interpretabile.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul-solutie. **Nu adauga si nu modifica niciun cod din afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi omise de grader, asa ca nu te baza pe celulele nou create pentru a gazdui codul-solutie; foloseste locurile deja oferite pentru asta.

* Evita sa folosesti variabile globale daca nu este absolut necesar. Grader-ul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Ca rezultat, variabilele globale pot fi indisponibile in momentul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai facand click pe iconita 💾 din coltul stanga-sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta-sus al paginii.
---


## Cuprins
- [Importuri](#imports)
- [1 - Introducere in Naive Bayes](#1)
    - [1.1 - Intelegerea teoremei lui Bayes](#1-1)
    - [1.2 - Presupunerea "naive"](#1-2)
- [2 - Gaussian Naive Bayes](#2)
    - **[Exercitiul 1 - calculate_statistics](#ex-1)**
    - **[Exercitiul 2 - gaussian_likelihood](#ex-2)**
    - **[Exercitiul 3 - predict_with_probabilities](#ex-3)**
- [3 - Aplicarea si analiza modelului](#3)
    - [3.1 - Incarcarea si pregatirea datelor](#3-1)
    - **[Exercitiul 4 - analyze_predictions](#ex-4)**
- [4 - Calibrarea probabilitatilor](#4)
    - **[Exercitiul 5 - calibrate_probabilities](#ex-5)**
- [5 - Multinomial Naive Bayes](#5)
    - **[Exercitiul 6 - multinomial_naive_bayes](#ex-6)**
- [6 - Analiza comparativa](#6)
    - **[Exercitiul 7 - compare_classifiers](#ex-7)**
- [7 - Clasificarea spam-ului](#7)
    - **[Exercitiul 8 - build_spam_classifier](#ex-8)**


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import unittests

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

<a name='1'></a>
## 1 - Introducere in Naive Bayes

Naive Bayes este un clasificator probabilistic bazat pe aplicarea teoremei lui Bayes impreuna cu presupunerea "naive" de independenta conditionata intre trasaturi. In ciuda simplitatii sale, este foarte eficient pentru multe aplicatii din lumea reala.


<a name='1-1'></a>
### 1.1 - Intelegerea teoremei lui Bayes

Teorema lui Bayes descrie probabilitatea unui eveniment pe baza cunostintelor anterioare despre conditiile asociate acelui eveniment:

$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

Unde:
- $P(y|X)$ este **probabilitatea posterioara**: probabilitatea clasei $y$ date fiind trasaturile $X$
- $P(X|y)$ este **verosimilitatea**: probabilitatea de a observa trasaturile $X$ data fiind clasa $y$
- $P(y)$ este **probabilitatea a priori**: probabilitatea clasei $y$ inainte de a vedea datele
- $P(X)$ este **evidenta**: probabilitatea de a observa trasaturile $X$

Deoarece comparam probabilitati intre clase pentru aceeasi instanta, $P(X)$ este constanta si poate fi ignorata in clasificare.


<a name='1-2'></a>
### 1.2 - Presupunerea "naive"

Presupunerea "naive" spune ca toate trasaturile sunt independente conditionat de clasa:

$$P(X|y) = P(x_1, x_2, ..., x_n|y) = \prod_{i=1}^{n} P(x_i|y)$$

Acest lucru simplifica dramatic calculul, deoarece trebuie sa calculam doar probabilitatile individuale ale trasaturilor, nu probabilitatile comune ale tuturor combinatiilor de trasaturi.

Predictia finala se face alegand clasa cu cea mai mare probabilitate posterioara:

$$\hat{y} = \arg\max_{y} P(y) \prod_{i=1}^{n} P(x_i|y)$$


<a name='2'></a>
## 2 - Gaussian Naive Bayes

Atunci cand lucram cu trasaturi continue, presupunem ca fiecare trasatura urmeaza o distributie Gaussiana (normala) in interiorul fiecarei clase. Functia de densitate de probabilitate pentru o distributie Gaussiana este:

$$P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

Unde $\mu_y$ si $\sigma_y^2$ sunt media si varianta trasaturii $x_i$ pentru clasa $y$.


<a name='ex-1'></a>
#### **Exercitiul 1 - `calculate_statistics`**

**Sarcina ta:**

Implementeaza functia `calculate_statistics` care calculeaza media, varianta si probabilitatile a priori pentru fiecare clasa din datele de antrenare. Aceste statistici reprezinta fundamentul lui Gaussian Naive Bayes.

Trebuie sa implementezi urmatoarele:

* **Calculeaza probabilitatile a priori ale claselor**:
    * Numarati cate exemple exista in fiecare clasa.
    * Impartiti la numarul total de exemple pentru a obtine $P(y)$.

* **Calculeaza media pentru fiecare trasatura din fiecare clasa**:
    * Pentru fiecare clasa, calculeaza media fiecarei trasaturi folosind exemplele care apartin acelei clase.
    * Stocheaza rezultatele intr-un dictionar care foloseste etichetele claselor drept chei.

* **Calculeaza varianta pentru fiecare trasatura din fiecare clasa**:
    * Pentru fiecare clasa, calculeaza varianta fiecarei trasaturi.
    * Adauga un epsilon mic ($10^{-9}$) pentru a preveni impartirea la zero.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor, iata un ghid mai detaliat:

**Pentru probabilitatile a priori ale claselor:**
* Foloseste `np.unique(y, return_counts=True)` pentru a obtine clasele unice si frecventele lor.
* Imparte frecventele la `len(y)` pentru a obtine probabilitatile.
* Stocheaza-le intr-un dictionar: `{class: probability}`.

**Pentru calculul mediei:**
* Itereaza prin fiecare clasa.
* Foloseste indexare booleana: `X[y == class_label]` pentru a obtine exemplele din acea clasa.
* Foloseste `np.mean(X_class, axis=0)` pentru a calcula media fiecarei trasaturi.

**Pentru calculul variantei:**
* Similar cu media, foloseste `X[y == class_label]` pentru fiecare clasa.
* Foloseste `np.var(X_class, axis=0)` pentru a calcula varianta.
* Adauga epsilon: `variance + 1e-9`.

</details>


In [ ]:
# GRADED FUNCTION: calculate_statistics
def calculate_statistics(X, y):
    """
    Calculate mean, variance, and prior probabilities for each class.
    
    Args:
        X: numpy array of shape (n_samples, n_features) - Training features
        y: numpy array of shape (n_samples,) - Training labels
    
    Returns:
        means: dict mapping class labels to mean vectors of shape (n_features,)
        variances: dict mapping class labels to variance vectors of shape (n_features,)
        priors: dict mapping class labels to prior probabilities (float)
    """
    means = {}
    variances = {}
    priors = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Get unique classes and their counts
    
    # Calculate prior probabilities for each class
    
    # Calculate mean and variance for each class
    
    ### SFÂRȘIT DE COD AICI ###
    
    return means, variances, priors

In [ ]:
# Test with a simple dataset
X_test = np.array([[1, 2], [2, 3], [3, 4], [6, 7], [7, 8], [8, 9]])
y_test = np.array([0, 0, 0, 1, 1, 1])

means, variances, priors = calculate_statistics(X_test, y_test)

print("Means per class:")
for class_label, mean in means.items():
    print(f"  Class {class_label}: {mean}")

print("\nVariances per class:")
for class_label, var in variances.items():
    print(f"  Class {class_label}: {var}")

print("\nPrior probabilities:")
for class_label, prior in priors.items():
    print(f"  Class {class_label}: {prior:.4f}")

##### **Rezultatul asteptat**
```
Means per class:
  Class 0: [2. 3.]
  Class 1: [7. 8.]

Variances per class:
  Class 0: [0.66666667 0.66666667]
  Class 1: [0.66666667 0.66666667]

Prior probabilities:
  Class 0: 0.5000
  Class 1: 0.5000
```


In [ ]:
unittests.exercise_1(calculate_statistics)

<a name='ex-2'></a>
#### **Exercitiul 2 - `gaussian_likelihood`**

**Sarcina ta:**

Implementeaza functia `gaussian_likelihood` care calculeaza probabilitatea de a observa o valoare a unei trasaturi data fiind o clasa, presupunand o distributie Gaussiana.

Trebuie sa implementezi:

* **Calculeaza densitatea de probabilitate Gaussiana**:
    * Foloseste formula: $P(x|y) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$
    * Calculeaza termenul exponentului: $-\frac{(x - \mu)^2}{2\sigma^2}$
    * Calculeaza coeficientul: $\frac{1}{\sqrt{2\pi\sigma^2}}$
    * Inmulteste-le pentru a obtine probabilitatea finala

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru formula Gaussiana:**
* Calculeaza exponentul: `exponent = -((x - mean) ** 2) / (2 * variance)`
* Calculeaza coeficientul: `coef = 1 / np.sqrt(2 * np.pi * variance)`
* Probabilitatea finala: `probability = coef * np.exp(exponent)`
* Foloseste `np.prod()` pentru a inmulti probabilitatile pe toate trasaturile

</details>


In [ ]:
# GRADED FUNCTION: gaussian_likelihood
def gaussian_likelihood(x, mean, variance):
    """
    Calculate the Gaussian probability density function for a given observation.
    
    Args:
        x: numpy array of shape (n_features,) - Feature values for a single sample
        mean: numpy array of shape (n_features,) - Mean values for each feature
        variance: numpy array of shape (n_features,) - Variance values for each feature
    
    Returns:
        likelihood: float - The joint probability of all features (product of individual probabilities)
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Calculate the exponent term
    
    # Calculate the coefficient
    
    # Calculate probability density for each feature
    
    # Return the product of all feature probabilities
    
    ### SFÂRȘIT DE COD AICI ###
    
    return likelihood

In [ ]:
# Test the gaussian_likelihood function
x_sample = np.array([2.0, 3.0])
mean_test = np.array([2.0, 3.0])
var_test = np.array([1.0, 1.0])

likelihood = gaussian_likelihood(x_sample, mean_test, var_test)
print(f"Likelihood when x equals mean: {likelihood:.6f}")

x_sample2 = np.array([3.0, 4.0])
likelihood2 = gaussian_likelihood(x_sample2, mean_test, var_test)
print(f"Likelihood when x differs from mean: {likelihood2:.6f}")

##### **Rezultatul asteptat**
```
Likelihood when x equals mean: 0.159155
Likelihood when x differs from mean: 0.058550
```


In [ ]:
unittests.exercise_2(gaussian_likelihood)

<a name='ex-3'></a>
#### **Exercitiul 3 - `predict_with_probabilities`**

**Sarcina ta:**

Implementeaza functia `predict_with_probabilities` care foloseste teorema lui Bayes pentru a prezice etichetele claselor impreuna cu scoruri de probabilitate pentru fiecare exemplu din setul de date.

Trebuie sa implementezi:

* **Calculeaza probabilitatile posterioare pentru fiecare clasa**:
    * Pentru fiecare exemplu, calculeaza probabilitatea posterioara pentru fiecare clasa.
    * Foloseste formula: $P(y|X) \propto P(y) \times P(X|y)$
    * Probabilitatea posterioara este probabilitatea a priori inmultita cu verosimilitatea.

* **Normalizeaza probabilitatile**:
    * Aduna toate probabilitatile posterioare intre clase.
    * Imparte fiecare probabilitate posterioara la aceasta suma pentru a obtine probabilitati normalizate.

* **Fa predictii**:
    * Alege clasa cu cea mai mare probabilitate posterioara.
    * Returneaza atat predictiile, cat si distributiile de probabilitate.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru calculul probabilitatilor:**
* Initializeaza un dictionar pentru a stoca probabilitatile posterioare ale fiecarei clase.
* Parcurge fiecare clasa si calculeaza: `prior * gaussian_likelihood(x, mean, variance)`.
* Stocheaza rezultatul in dictionar.

**Pentru normalizare:**
* Aduna toate valorile posterioare: `total = sum(posteriors.values())`.
* Imparte fiecare probabilitate posterioara la total: `{class: prob/total for class, prob in posteriors.items()}`.

**Pentru predictie:**
* Foloseste `max(posteriors, key=posteriors.get)` pentru a gasi clasa cu probabilitatea cea mai mare.
* Stocheaza predictiile intr-o lista si probabilitatile intr-o alta lista sau matrice.

</details>


In [ ]:
# GRADED FUNCTION: predict_with_probabilities
def predict_with_probabilities(X, means, variances, priors):
    """
    Predict class labels and return probability distributions for each sample.
    
    Args:
        X: numpy array of shape (n_samples, n_features) - Samples to predict
        means: dict mapping class labels to mean vectors
        variances: dict mapping class labels to variance vectors
        priors: dict mapping class labels to prior probabilities
    
    Returns:
        predictions: numpy array of shape (n_samples,) - Predicted class labels
        probabilities: list of dicts - Each dict maps class labels to probabilities for that sample
    """
    predictions = []
    probabilities = []
    
    ### ÎNCEPUT DE COD AICI ###
    
    # For each sample in X
    
        # Calculate posterior probability for each class
        
        # Normalize probabilities
        
        # Predict class with highest probability
        
    ### SFÂRȘIT DE COD AICI ###
    
    return np.array(predictions), probabilities

In [ ]:
# Test the predict_with_probabilities function
X_test_pred = np.array([[2, 3], [7, 8], [4, 5]])

predictions, probs = predict_with_probabilities(X_test_pred, means, variances, priors)

print("Predictions:", predictions)
print("\nProbability distributions:")
for i, prob_dist in enumerate(probs):
    print(f"Sample {i}: {prob_dist}")

##### **Rezultatul asteptat**
```
Predictions: [0 1 0]

Probability distributions:
Sample 0: {0: 0.8807970779778824, 1: 0.11920292202211755}
Sample 1: {0: 0.11920292202211755, 1: 0.8807970779778824}
Sample 2: {0: 0.7310585786300049, 1: 0.2689414213699951}
```


In [ ]:
unittests.exercise_3(predict_with_probabilities)

<a name='3'></a>
## 3 - Aplicarea si analiza modelului

Acum ca ai implementat algoritmul de baza Gaussian Naive Bayes, hai sa il aplicam pe seturi de date din lumea reala si sa ii analizam performanta.


<a name='3-1'></a>
### 3.1 - Incarcarea si pregatirea datelor

Vom folosi setul de date Iris, un set de date clasic din machine learning, cu trasaturi continue care reprezinta masuratori ale florilor.


In [ ]:
# Load the Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y_train))}")

In [ ]:
# Train the Gaussian Naive Bayes model
means_iris, variances_iris, priors_iris = calculate_statistics(X_train, y_train)

# Make predictions
y_pred, y_probs = predict_with_probabilities(X_test, means_iris, variances_iris, priors_iris)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

<a name='ex-4'></a>
#### **Exercitiul 4 - `analyze_predictions`**

**Sarcina ta:**

Implementeaza functia `analyze_predictions` care analizeaza calitatea predictiilor examinind nivelurile de incredere si identificind predictiile incerte.

Trebuie sa implementezi:

* **Calculeaza scorurile de incredere**:
    * Pentru fiecare predictie, increderea este probabilitatea maxima dintre toate clasele.
    * Extrage aceasta informatie din distributiile de probabilitate returnate de `predict_with_probabilities`.

* **Identifica predictiile cu incredere mare si mica**:
    * Sorteaza predictiile dupa nivelul de incredere.
    * Gaseste primele N predictii cele mai sigure.
    * Gaseste primele N predictii cele mai putin sigure (cele mai incerte).

* **Calculeaza statistici**:
    * Calculeaza increderea medie pentru predictiile corecte si pentru cele incorecte.
    * Numarati cate predictii se afla sub un prag de incredere (de exemplu, 0.5).

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru scorurile de incredere:**
* Extrage probabilitatea maxima pentru fiecare exemplu: `confidence = max(prob_dict.values())`.
* Stocheaza rezultatele intr-o lista sau un array.

**Pentru incredere mare/mica:**
* Creeaza o lista de tupluri `(index, confidence)`.
* Sorteaza dupa incredere: `sorted(items, key=lambda x: x[1])`.
* Ia primele N pentru cele mai mici si ultimele N pentru cele mai mari.

**Pentru statistici:**
* Foloseste indexare booleana: `confidences[predictions == y_true]` pentru predictiile corecte.
* Calculeaza media: `np.mean(correct_confidences)`.
* Numarati predictiile cu incredere mica: `sum(confidences < threshold)`.

</details>


In [ ]:
# GRADED FUNCTION: analyze_predictions
def analyze_predictions(y_true, y_pred, probabilities, top_n=5):
    """
    Analyze prediction quality by examining confidence levels.
    
    Args:
        y_true: numpy array of shape (n_samples,) - True labels
        y_pred: numpy array of shape (n_samples,) - Predicted labels
        probabilities: list of dicts - Probability distributions for each sample
        top_n: int - Number of top confident/uncertain predictions to return
    
    Returns:
        analysis: dict containing:
            - 'confidences': numpy array of confidence scores
            - 'avg_confidence': float - Overall average confidence
            - 'avg_confidence_correct': float - Average confidence for correct predictions
            - 'avg_confidence_incorrect': float - Average confidence for incorrect predictions
            - 'most_confident': list of (index, confidence) tuples
            - 'least_confident': list of (index, confidence) tuples
            - 'low_confidence_count': int - Count of predictions below 0.5 confidence
    """
    analysis = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Extract confidence scores (max probability for each sample)
    
    # Calculate average confidence for correct predictions
    
    # Calculate average confidence for incorrect predictions
    
    # Find most and least confident predictions
    
    # Count predictions with confidence below 0.5
    
    ### SFÂRȘIT DE COD AICI ###
    
    return analysis

In [ ]:
# Test the analyze_predictions function
analysis_results = analyze_predictions(y_test, y_pred, y_probs, top_n=3)

print(f"Average confidence (correct): {analysis_results['avg_confidence_correct']:.4f}")
print(f"Average confidence (incorrect): {analysis_results['avg_confidence_incorrect']:.4f}")
print(f"\nLow confidence predictions (< 0.5): {analysis_results['low_confidence_count']}")

print("\nMost confident predictions:")
for idx, conf in analysis_results['most_confident']:
    print(f"  Sample {idx}: confidence = {conf:.4f}, true = {y_test[idx]}, pred = {y_pred[idx]}")

print("\nLeast confident predictions:")
for idx, conf in analysis_results['least_confident']:
    print(f"  Sample {idx}: confidence = {conf:.4f}, true = {y_test[idx]}, pred = {y_pred[idx]}")

##### **Rezultatul asteptat** (valorile pot varia usor)
```
Average confidence (correct): 0.9700
Average confidence (incorrect): 0.7500

Low confidence predictions (< 0.5): 0

Most confident predictions:
  Sample X: confidence = 1.0000, true = Y, pred = Y
  ...

Least confident predictions:
  Sample X: confidence = 0.XXXX, true = Y, pred = Z
  ...
```


In [ ]:
unittests.exercise_4(analyze_predictions)

<a name='4'></a>
## 4 - Calibrarea probabilitatilor

Desi Naive Bayes ofera estimari de probabilitate, aceste probabilitati adesea nu sunt bine calibrate. Calibrarea ajusteaza probabilitatile prezise astfel incat sa reflecte mai bine probabilitatea reala ca predictia sa fie corecta.

Un clasificator bine calibrat ar trebui sa aiba proprietatea ca, dintre toate predictiile cu incredere 0.8, aproximativ 80% sunt corecte.


<a name='ex-5'></a>
#### **Exercitiul 5 - `calibrate_probabilities`**

**Sarcina ta:**

Implementeaza functia `calibrate_probabilities` folosind scalarea Platt (calibrare logistica). Aceasta tehnica antreneaza un model de regresie logistica pe probabilitatile prezise pentru a produce probabilitati calibrate.

Trebuie sa implementezi:

* **Extrage scorurile de incredere**:
    * Obtine probabilitatea maxima pentru fiecare predictie.
    * Acestea vor fi trasaturile pentru modelul de calibrare.

* **Antreneaza o calibrare logistica simpla**:
    * Creeaza etichete binare: 1 daca predictia este corecta, 0 altfel.
    * Potriveste o transformare logistica: $p_{calibrated} = \frac{1}{1 + e^{-(a \cdot p + b)}}$
    * Poti folosi o abordare simplificata sau `LogisticRegression` din sklearn.

* **Aplica calibrarea**:
    * Transforma scorurile de incredere folosind parametrii invatati.
    * Returneaza probabilitatile calibrate.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru extragerea scorurilor:**
* Parcurge probabilitatile: `scores = [max(p.values()) for p in probabilities]`.

**Pentru etichetele binare:**
* Creeaza etichetele de corectitudine: `correct = (y_pred == y_true).astype(int)`.

**Pentru calibrare (abordare simplificata):**
* Foloseste `LogisticRegression` din sklearn: `from sklearn.linear_model import LogisticRegression`.
* Antreneaza pe scorurile de incredere: `model.fit(scores.reshape(-1, 1), correct)`.
* Prezice probabilitatile calibrate: `calibrated = model.predict_proba(scores.reshape(-1, 1))[:, 1]`.

**Alternativa: Implementare manuala:**
* Calculeaza probabilitatea medie si rata medie de corectitudine pe intervale.
* Aplica regresie izotonica sau histogram binning.

</details>


In [ ]:
# GRADED FUNCTION: calibrate_probabilities
def calibrate_probabilities(y_true, y_pred, probabilities):
    """
    Calibrate predicted probabilities using Platt scaling.
    
    Args:
        y_true: numpy array of shape (n_samples,) - True labels
        y_pred: numpy array of shape (n_samples,) - Predicted labels
        probabilities: list of dicts OR numpy array of confidence scores

    Returns:
        calibrated_probs: numpy array of shape (n_samples,) - Calibrated confidence scores
        calibration_model: fitted calibration model (for applying to new data)
    """
    from sklearn.linear_model import LogisticRegression
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Extract confidence scores (max probability for each sample)
    
    # Create binary correctness labels
    
    # Fit logistic calibration model
    
    # Apply calibration to get calibrated probabilities
    
    ### SFÂRȘIT DE COD AICI ###
    
    return calibrated_probs, calibration_model

In [ ]:
# Test the calibrate_probabilities function
calibrated_probs, calib_model = calibrate_probabilities(y_test, y_pred, y_probs)

print("Original vs Calibrated Probabilities (first 10 samples):")
print("Original | Calibrated | Correct?")
print("-" * 40)
for i in range(min(10, len(y_pred))):
    original_conf = max(y_probs[i].values())
    correct = "✓" if y_pred[i] == y_test[i] else "✗"
    print(f"{original_conf:.4f}   | {calibrated_probs[i]:.4f}     | {correct}")

##### **Rezultatul asteptat** (valorile vor varia)
```
Original vs Calibrated Probabilities (first 10 samples):
Original | Calibrated | Correct?
----------------------------------------
0.9999   | 0.9856     | ✓
0.9856   | 0.9745     | ✓
...
```


In [ ]:
unittests.exercise_5(calibrate_probabilities)

<a name='5'></a>
## 5 - Multinomial Naive Bayes

Pana acum, am lucrat cu trasaturi continue folosind Gaussian Naive Bayes. Acum vom implementa **Multinomial Naive Bayes**, care este conceput pentru trasaturi discrete, precum numararile de cuvinte din clasificarea textelor.

In Multinomial Naive Bayes, trasaturile reprezinta numarari (de exemplu, frecvente ale cuvintelor), iar verosimilitatea este modelata folosind distributia multinomiala:

$$P(x_i|y) = \frac{N_{yi} + \alpha}{N_y + \alpha n}$$

Unde:
- $N_{yi}$ este numarul de aparitii al trasaturii $i$ in clasa $y$
- $N_y$ este numarul total de aparitii ale tuturor trasaturilor in clasa $y$
- $\alpha$ este parametrul de netezire (de obicei 1, numit Laplace smoothing)
- $n$ este numarul de trasaturi


<a name='ex-6'></a>
#### **Exercitiul 6 - `multinomial_naive_bayes`**

**Sarcina ta:**

Implementeaza un clasificator complet Multinomial Naive Bayes care poate invata pe datele de antrenare si poate prezice pe datele de test.

Trebuie sa implementezi:

* **Metoda fit - calculeaza probabilitatile trasaturilor**:
    * Pentru fiecare clasa, numara aparitiile fiecarei valori de trasatura.
    * Aplica netezirea Laplace pentru a evita probabilitatile zero.
    * Calculeaza probabilitatile a priori pentru fiecare clasa.

* **Metoda predict - face predictii**:
    * Pentru fiecare exemplu de test, calculeaza log-probabilitatile pentru fiecare clasa.
    * Foloseste log-probabilitati pentru a evita underflow-ul numeric: $\log P(y|X) = \log P(y) + \sum_i \log P(x_i|y)$
    * Alege clasa cu log-probabilitatea cea mai mare.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru metoda fit:**
* Calculeaza probabilitatile a priori ale claselor: `priors[c] = (y == c).sum() / len(y)`.
* Pentru fiecare clasa, aduna numararile trasaturilor: `feature_counts = X[y == c].sum(axis=0)`.
* Aplica netezirea Laplace: `(feature_counts + alpha) / (feature_counts.sum() + alpha * n_features)`.
* Stocheaza intr-un dictionar: `self.feature_log_probs[class] = np.log(probabilities)`.

**Pentru metoda predict:**
* Calculeaza log-verosimilitatea: `log_likelihood = X @ self.feature_log_probs[class]`.
* Adauga log-priorul: `log_posterior = log_prior + log_likelihood`.
* Alege clasa cu log-posteriorul maxim: `np.argmax(log_posteriors, axis=1)`.

</details>


In [ ]:
# GRADED FUNCTION: MultinomialNB class
class MultinomialNB:
    """
    Multinomial Naive Bayes classifier for discrete features.
    """
    def __init__(self, alpha=1.0):
        """
        Initialize the classifier.
        
        Args:
            alpha: float - Smoothing parameter (Laplace smoothing)
        """
        self.alpha = alpha
        self.classes = None
        self.feature_log_probs = {}
        self.class_log_priors = {}
    
    def fit(self, X, y):
        """
        Fit the Multinomial Naive Bayes model.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Training features (counts)
            y: numpy array of shape (n_samples,) - Training labels
        """
        ### ÎNCEPUT DE COD AICI ###
        
        # Get unique classes
        
        # Calculate class priors
        
        # For each class, calculate feature probabilities with Laplace smoothing
        
        ### SFÂRȘIT DE COD AICI ###
        
        return self
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Test features
        
        Returns:
            predictions: numpy array of shape (n_samples,) - Predicted labels
        """
        ### ÎNCEPUT DE COD AICI ###
        
        # Calculate log posterior for each class
        
        # Choose class with highest log posterior
        
        ### SFÂRȘIT DE COD AICI ###
        
        return predictions
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Test features
        
        Returns:
            probabilities: numpy array of shape (n_samples, n_classes) - Class probabilities
        """
        ### ÎNCEPUT DE COD AICI ###
        
        # Calculate log posteriors
        
        # Convert to probabilities using softmax
        
        ### SFÂRȘIT DE COD AICI ###
        
        return probabilities

In [ ]:
# Test with a simple text-like dataset
# Create a toy dataset with word counts
X_text = np.array([
    [3, 0, 1, 0, 0, 5],  # Document 1 (class 0)
    [2, 0, 0, 1, 0, 4],  # Document 2 (class 0)
    [0, 4, 0, 0, 3, 1],  # Document 3 (class 1)
    [0, 3, 0, 1, 2, 0],  # Document 4 (class 1)
    [4, 0, 2, 0, 0, 6],  # Document 5 (class 0)
    [0, 5, 0, 0, 4, 0],  # Document 6 (class 1)
])
y_text = np.array([0, 0, 1, 1, 0, 1])

# Fit the model
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_text, y_text)

# Make predictions
X_test_text = np.array([
    [3, 0, 1, 0, 0, 4],  # Similar to class 0
    [0, 4, 0, 0, 3, 0],  # Similar to class 1
])

predictions = mnb.predict(X_test_text)
probabilities = mnb.predict_proba(X_test_text)

print("Predictions:", predictions)
print("\nProbabilities:")
print(probabilities)

##### **Rezultatul asteptat**
```
Predictions: [0 1]

Probabilities:
[[0.99... 0.00...]
 [0.00... 0.99...]]
```


In [ ]:
unittests.exercise_6(MultinomialNB)

<a name='6'></a>
## 6 - Analiza comparativa

Acum sa comparam Naive Bayes cu Regresia Logistica pentru a intelege punctele forte si slabiciunile fiecarui algoritm.


<a name='ex-7'></a>
#### **Exercitiul 7 - `compare_classifiers`**

**Sarcina ta:**

Implementeaza functia `compare_classifiers` care antreneaza atat Naive Bayes, cat si Regresia Logistica pe acelasi set de date si compara performanta lor pe mai multe metrici.

Trebuie sa implementezi:

* **Antreneaza ambele modele**:
    * Foloseste implementarea ta de Gaussian Naive Bayes.
    * Foloseste `LogisticRegression` din sklearn pentru comparatie.

* **Fa predictii**:
    * Obtine predictii si probabilitati de la ambele modele.

* **Calculeaza metrici**:
    * Acuratete pentru ambele modele.
    * Timp de antrenare pentru ambele modele.
    * Increderea medie a predictiilor pentru ambele modele.

* **Returneaza dictionarul de comparatie**:
    * Include toate metricele pentru o comparatie usoara.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru Naive Bayes:**
* Foloseste functiile pe care le-ai implementat deja: `calculate_statistics` si `predict_with_probabilities`.
* Cronometreaza antrenarea: `import time; start = time.time(); ...; duration = time.time() - start`.

**Pentru Regresia Logistica:**
* Import: `from sklearn.linear_model import LogisticRegression`.
* Fit: `model = LogisticRegression().fit(X_train, y_train)`.
* Predictii: `y_pred = model.predict(X_test)` si `y_proba = model.predict_proba(X_test)`.

**Pentru metrici:**
* Acuratete: `accuracy_score(y_test, y_pred)`.
* Incredere: `np.max(probabilities, axis=1).mean()` pentru output-ul sklearn.
* Pentru NB: `np.array([max(p.values()) for p in nb_probs]).mean()`.

</details>


In [ ]:
# GRADED FUNCTION: compare_classifiers
def compare_classifiers(X_train, X_test, y_train, y_test):
    """
    Compare Gaussian Naive Bayes with Logistic Regression.
    
    Args:
        X_train: numpy array - Training features
        X_test: numpy array - Test features
        y_train: numpy array - Training labels
        y_test: numpy array - Test labels
    
    Returns:
        comparison: dict containing:
            - 'nb_accuracy': float
            - 'lr_accuracy': float
            - 'nb_train_time': float (in seconds)
            - 'lr_train_time': float (in seconds)
            - 'nb_avg_confidence': float
            - 'lr_avg_confidence': float
            - 'nb_predictions': numpy array
            - 'lr_predictions': numpy array
    """
    import time
    from sklearn.linear_model import LogisticRegression
    
    comparison = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Train Naive Bayes
    
    # Make NB predictions
    
    # Calculate NB metrics
    
    # Train Logistic Regression
    
    # Make LR predictions
    
    # Calculate LR metrics
    
    ### SFÂRȘIT DE COD AICI ###
    
    return comparison

In [ ]:
# Compare classifiers on Iris dataset
comparison_results = compare_classifiers(X_train, X_test, y_train, y_test)

print("=" * 60)
print("CLASSIFIER COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<30} {'Naive Bayes':<15} {'Logistic Regression':<15}")
print("-" * 60)
print(f"{'Accuracy':<30} {comparison_results['nb_accuracy']:<15.4f} {comparison_results['lr_accuracy']:<15.4f}")
print(f"{'Training Time (s)':<30} {comparison_results['nb_train_time']:<15.6f} {comparison_results['lr_train_time']:<15.6f}")
print(f"{'Average Confidence':<30} {comparison_results['nb_avg_confidence']:<15.4f} {comparison_results['lr_avg_confidence']:<15.4f}")
print("=" * 60)

# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_nb = confusion_matrix(y_test, comparison_results['nb_predictions'])
cm_lr = confusion_matrix(y_test, comparison_results['lr_predictions'])

axes[0].imshow(cm_nb, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('Naive Bayes Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

axes[1].imshow(cm_lr, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Logistic Regression Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

##### **Rezultatul asteptat** (valorile vor varia)
```
============================================================
CLASSIFIER COMPARISON
============================================================

Metric                         Naive Bayes     Logistic Regression
------------------------------------------------------------
Accuracy                       0.9778          0.9778
Training Time (s)              0.000XXX        0.00XXXX
Average Confidence             0.96XX          0.97XX
============================================================
```


In [ ]:
unittests.exercise_7(compare_classifiers)

<a name='7'></a>
## 7 - Clasificarea spam-ului

In final, sa construim un clasificator complet de spam folosind Multinomial Naive Bayes. Aceasta este una dintre cele mai comune aplicatii ale lui Naive Bayes in lumea reala.


<a name='ex-8'></a>
#### **Exercitiul 8 - `build_spam_classifier`**

**Sarcina ta:**

Implementeaza functia `build_spam_classifier` care creeaza un pipeline complet de detectare a spam-ului, cu extragere de trasaturi, antrenarea modelului si scoruri de probabilitate interpretabile.

Trebuie sa implementezi:

* **Extragerea trasaturilor**:
    * Converteste mesajele text in trasaturi bazate pe numararea cuvintelor.
    * Foloseste o abordare simpla bag-of-words sau `CountVectorizer`.
    * Creeaza un vocabular al celor mai frecvente cuvinte.

* **Antreneaza Multinomial Naive Bayes**:
    * Foloseste clasa ta `MultinomialNB` pe matricea de trasaturi.

* **Fa predictii cu probabilitati**:
    * Clasifica mesajele ca spam sau ham (non-spam).
    * Returneaza scoruri de probabilitate pentru interpretabilitate.

* **Identifica principalii indicatori de spam**:
    * Gaseste cuvintele care au cele mai mari rapoarte de probabilitate pentru spam fata de ham.
    * Acest lucru ajuta la explicarea motivului pentru care mesajele sunt clasificate ca spam.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru extragerea trasaturilor:**
* Foloseste `CountVectorizer` din sklearn: `from sklearn.feature_extraction.text import CountVectorizer`.
* Antreneaza pe datele de train: `vectorizer = CountVectorizer(max_features=100).fit(X_train)`.
* Transforma atat train, cat si test: `X_train_counts = vectorizer.transform(X_train).toarray()`.

**Pentru antrenare:**
* Foloseste clasa ta `MultinomialNB`: `model = MultinomialNB().fit(X_train_counts, y_train)`.

**Pentru indicatorii de spam:**
* Compara log-probabilitatile trasaturilor intre clase.
* Calculeaza log-ratio: `log_ratio = feature_log_probs[spam] - feature_log_probs[ham]`.
* Sorteaza dupa raport pentru a gasi principalele cuvinte de spam: `np.argsort(log_ratio)[-10:]`.
* Mapeaza indicii inapoi la cuvinte folosind vectorizer-ul: `vectorizer.get_feature_names_out()`.

</details>


In [ ]:
# GRADED FUNCTION: build_spam_classifier
def build_spam_classifier(X_train, X_test, y_train, y_test, max_features=100):
    """
    Build a complete spam classifier with feature extraction and interpretation.

    Args:
        X_train: list of strings OR numeric array - Training messages / features
        X_test: list of strings OR numeric array - Test messages / features
        y_train: numpy array - Training labels (0=ham, 1=spam)
        y_test: numpy array - Test labels
        max_features: int - Maximum number of features (used only for text input)

    Returns:
        results: dict containing:
            - 'model': trained MultinomialNB model
            - 'vectorizer': fitted CountVectorizer (or None for numeric input)
            - 'accuracy': float - Test accuracy
            - 'predictions': numpy array - Predicted labels
            - 'probabilities': numpy array - Prediction probabilities
            - 'top_spam_words': list of (word, score) tuples
    """
    from sklearn.feature_extraction.text import CountVectorizer

    results = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Feature extraction
    
    # Train model
    
    # Make predictions
    
    # Calculate accuracy
    
    # Find top spam indicator words
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Create a toy spam dataset
spam_messages = [
    "Free money now! Click here to win!",
    "Congratulations! You've won a prize. Claim now!",
    "Buy now! Limited time offer! Free shipping!",
    "Win big! Click this link for free money!",
    "Get rich quick! Free trial available now!",
]

ham_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Can you send me the report by end of day?",
    "Thanks for your help with the project.",
    "Let me know when you're available to chat.",
    "I'll send you the documents this afternoon.",
]

# Combine and create labels
X_spam = spam_messages + ham_messages
y_spam = np.array([1]*5 + [0]*5)

# Create train/test split
X_train_spam = X_spam[:8]
X_test_spam = X_spam[8:]
y_train_spam = y_spam[:8]
y_test_spam = y_spam[8:]

# Build spam classifier
spam_results = build_spam_classifier(X_train_spam, X_test_spam, y_train_spam, y_test_spam, max_features=50)

print(f"Spam Classifier Accuracy: {spam_results['accuracy']:.4f}")
print(f"\nTest Predictions: {spam_results['predictions']}")
print(f"True Labels: {y_test_spam}")

print("\nTop Spam Indicator Words:")
for word, score in spam_results['top_spam_words'][:10]:
    print(f"  {word}: {score:.4f}")

# Test on new messages
print("\n" + "="*60)
print("TESTING ON NEW MESSAGES")
print("="*60)

new_messages = [
    "Free money and prizes! Click now!",
    "Can we schedule a meeting next week?",
]

new_features = spam_results['vectorizer'].transform(new_messages).toarray()
new_predictions = spam_results['model'].predict(new_features)
new_probs = spam_results['model'].predict_proba(new_features)

for msg, pred, prob in zip(new_messages, new_predictions, new_probs):
    label = "SPAM" if pred == 1 else "HAM"
    confidence = prob[pred]
    print(f"\nMessage: '{msg}'")
    print(f"Prediction: {label} (confidence: {confidence:.4f})")

##### **Rezultatul asteptat** (cuvintele exacte si scorurile pot varia)
```
Spam Classifier Accuracy: 1.0000

Test Predictions: [1 0]
True Labels: [1 0]

Top Spam Indicator Words:
  free: X.XXXX
  money: X.XXXX
  now: X.XXXX
  ...

============================================================
TESTING ON NEW MESSAGES
============================================================

Message: 'Free money and prizes! Click now!'
Prediction: SPAM (confidence: 0.XXXX)

Message: 'Can we schedule a meeting next week?'
Prediction: HAM (confidence: 0.XXXX)
```


In [ ]:
unittests.exercise_8(build_spam_classifier)

## Felicitari!

Ai finalizat cu succes tema despre Naive Bayes! Ai construit o intelegere cuprinzatoare a unuia dintre cei mai eleganti si mai interpretabili algoritmi de machine learning.

### Ce ai realizat:

1. **Ai inteles teorema lui Bayes**: Ai invatat fundamentul matematic al clasificarii probabilistice
2. **Ai implementat Gaussian Naive Bayes**: Ai construit de la zero un clasificator pentru trasaturi continue
3. **Ai calculat probabilitati de verosimilitate**: Ai folosit distributii Gaussiene pentru a modela probabilitatile trasaturilor
4. **Ai facut predictii cu probabilitati**: Ai aplicat teorema lui Bayes pentru a genera predictii interpretabile
5. **Ai implementat calibrarea probabilitatilor**: Ai imbunatatit increderea predictiilor folosind scalarea Platt
6. **Ai construit Multinomial Naive Bayes**: Ai creat un clasificator pentru trasaturi discrete si date text
7. **Ai comparat clasificatori**: Ai analizat cand exceleaza Naive Bayes fata de Regresia Logistica
8. **Ai creat un clasificator de spam**: Ai construit o aplicatie completa din lumea reala, cu rezultate interpretabile

### Idei-cheie:

* **Naive Bayes este rapid si eficient**: Perfect pentru aplicatii in timp real si seturi mari de date
* **Interpretabilitatea conteaza**: Scorurile de probabilitate ajuta la explicarea deciziilor modelului
* **Presupunerea de independenta a trasaturilor**: Desi este "naive", functioneaza remarcabil de bine in practica
* **Clasificarea textelor**: Multinomial Naive Bayes exceleaza in clasificarea documentelor si a spam-ului
* **Calibrarea probabilitatilor**: Probabilitatile brute pot fi imbunatatite pentru o luare a deciziilor mai buna

**Foarte bine! Acum ai o baza solida in machine learning probabilistic!**
